In [9]:
import pandas as pd
import numpy as np
import pymysql
from sqlalchemy import create_engine
import getpass  
password = getpass.getpass()

In [24]:
bd = "sakila"
connection_string = 'mysql+pymysql://root:' + password + '@localhost/'+bd
engine = create_engine(connection_string)
engine
def rentals_month(engine, month: int, year: int) -> pd.DataFrame:
    sql = text("""
        SELECT rental_id, rental_date, inventory_id, customer_id, return_date, staff_id
        FROM rental
        WHERE YEAR(rental_date) = :year AND MONTH(rental_date) = :month
        ORDER BY rental_id
    """)
    with engine.connect() as conn:
        df = pd.read_sql_query(sql, con=conn, params={"year": year, "month": month},
                               parse_dates=["rental_date", "return_date"])
    return df
df = rentals_month(engine, 5, 2005)
print(df.head())

   rental_id         rental_date  inventory_id  customer_id  \
0          1 2005-05-24 22:53:30           367          130   
1          2 2005-05-24 22:54:33          1525          459   
2          3 2005-05-24 23:03:39          1711          408   
3          4 2005-05-24 23:04:41          2452          333   
4          5 2005-05-24 23:05:21          2079          222   

          return_date  staff_id  
0 2005-05-26 22:04:30         1  
1 2005-05-28 19:40:33         1  
2 2005-06-01 22:12:39         1  
3 2005-06-03 01:43:41         2  
4 2005-06-02 04:33:21         1  


In [28]:

def rental_count_month(rentals_df: pd.DataFrame, month: int, year: int) -> pd.DataFrame:

    month = int(month)
    year = int(year)
    col_name = f"rentals_{month:02d}_{year}"

    df = rentals_df.copy()

    if "rental_date" in df.columns:
        df["rental_date"] = pd.to_datetime(df["rental_date"])
        df = df[(df["rental_date"].dt.year == year) & (df["rental_date"].dt.month == month)]

    count_col = "rental_id" if "rental_id" in df.columns else df.columns[0]

    result = (
        df.groupby("customer_id", dropna=False)
          .agg(**{col_name: (count_col, "count")})
          .reset_index()
          .sort_values(by=[col_name, "customer_id"], ascending=[False, True])
          .reset_index(drop=True)
    )

    return result


In [29]:
df = rentals_month(engine, 5, 2005)
counts = rental_count_month(df, 5, 2005)
print(counts.head())

   customer_id  rentals_05_2005
0          197                8
1          109                7
2          506                7
3           19                6
4           53                6


In [ ]:
def compare_rentals(df1: pd.DataFrame, df2: pd.DataFrame):

    def find_rent_col(df: pd.DataFrame) -> str:
        cols = [c for c in df.columns if c != "customer_id"]
        for c in cols:
            if pd.api.types.is_numeric_dtype(df[c]):
                return c
        raise ValueError("No numeric rentals column found")

    col1 = find_rent_col(df1)
    col2 = find_rent_col(df2)

    merged = pd.merge(
        df1[["customer_id", col1]],
        df2[["customer_id", col2]],
        on="customer_id",
        how="outer"
    )

    merged[[col1, col2]] = merged[[col1, col2]].fillna(0).astype(int)
    merged["difference"] = merged[col2] - merged[col1]
    merged = merged.sort_values(by=["difference", "customer_id"], ascending=[False, True]).reset_index(drop=True)

    return merged


In [33]:
m1, y1 = 5, 2005
m2, y2 = 6, 2005
rentals1 = rentals_month(engine, m1, y1)
rentals2 = rentals_month(engine, m2, y2)
counts1 = rental_count_month(rentals1, m1, y1)
counts2 = rental_count_month(rentals2, m2, y2)
comparison = compare_rentals(counts1, counts2)

print(comparison.head())

   customer_id  rentals_05_2005  rentals_06_2005  difference
0           31                0               11          11
1          329                0                9           9
2          454                1               10           9
3          178                0                8           8
4          213                1                9           8
